# Datenanalyse mit SQL & Python - Tag 2: Joins, HAVING, Subqueries und CTEs

**Thema:** Shop-Daten mit mehreren Tabellen analysieren  
**Datensatz:** `shop_orders`, `shop_customers`, `shop_products`  
**Ziel:** Tag-1-SQL wiederholen und Schritt für Schritt zu Joins, `HAVING`, Unterabfragen und CTEs erweitern.


## Ablauf und Zeitplan

| Zeit | Kursphase | Inhalt |
|---|---|---|
| 09:00-09:30 | Wiederholung | Übung: `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `COUNT`, `SUM`, `AVG`, `MAX` |
| 09:30-10:30 | `GROUP BY`, `HAVING` & `JOINS` | `GROUP BY` & `HAVING`, `JOINS` & Star-Schema |
| 10:30-11:30 | Subqueries & Datenmodellierung | Subqueries in `WHERE` / `FROM`, Primärschlüssel & Fremdschlüssel, Grundidee von relationalen Modellen und Star-Schema-Logik |
| 11:30-12:10 | Mittagspause | - |
| 12:10-14:00 | Erweiterte SQL-Konzepte & Übung | Common Table Expressions (`CTE`), Window Functions, z. B. Ranking und laufende Summen |
| 14:00-14:20 | Pause | - |
| 14:20-15:40 | Praktische Übung | Mini-Projekt & Präsentation vorbereiten |
| 15:40-16:00 | Ergebnisse & Diskussion | Präsentation, Reflexion, Q&A |

**Hinweis zur Zeitplanung:** Die Zeiten sind Richtwerte. Wenn eine Diskussion besonders hilfreich ist, kann ein Block leicht angepasst werden.


## Lernziele

Nach Tag 2 kannst du:

- `GROUP BY` aus Tag 1 sicher wiederholen
- `WHERE` und `HAVING` unterscheiden
- Bestellungen mit Kundendaten per `JOIN` verbinden
- mehrere Tabellen kombinieren, um fachliche Kennzahlen zu berechnen
- einfache Unterabfragen lesen und schreiben
- einfache CTEs mit `WITH` verwenden und mit Unterabfragen vergleichen


## Datenmodell: Shop

Wir arbeiten mit drei Tabellen:

| Tabelle | Inhalt | Wichtige Spalten |
|---|---|---|
| `orders` | Bestellungen | `order_id`, `customer_id`, `product_id`, `quantity` |
| `customers` | Kundinnen und Kunden | `customer_id`, `name`, `city` |
| `products` | Produkte | `product_id`, `product_name`, `category`, `price` |

Beziehungen:

- `orders.customer_id` verweist auf `customers.customer_id`
- `orders.product_id` verweist auf `products.product_id`


## Einrichtung & Daten laden

Diese Zelle lädt die Shop-Daten und erstellt daraus eine SQLite-Datenbank im Arbeitsspeicher.


In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


def find_shop_dir():
    candidates = [
        Path('data_practice_tables/shop'),
        Path('Tag 1/data_practice_tables/shop'),
        Path('../Tag 1/data_practice_tables/shop'),
    ]
    for candidate in candidates:
        if (candidate / 'shop_orders.csv').exists():
            return candidate
    raise FileNotFoundError('Shop-Daten wurden nicht gefunden.')

SHOP_DIR = find_shop_dir()

orders_df = pd.read_csv(SHOP_DIR / 'shop_orders.csv')
customers_df = pd.read_csv(SHOP_DIR / 'shop_customers.csv')
products_df = pd.read_csv(SHOP_DIR / 'shop_products.csv')

conn = sqlite3.connect(':memory:')
orders_df.to_sql('orders', conn, index=False, if_exists='replace')
customers_df.to_sql('customers', conn, index=False, if_exists='replace')
products_df.to_sql('products', conn, index=False, if_exists='replace')

print('Geladene Tabellen:')
print('orders:   ', orders_df.shape)
print('customers:', customers_df.shape)
print('products: ', products_df.shape)


## 1. Warm-up: Orders-Tabelle verstehen

Wir starten nur mit `orders`. Das wiederholt Tag 1 und zeigt gleichzeitig, warum wir später Joins brauchen.


In [ ]:
query = '''
SELECT *
FROM orders;
'''

pd.read_sql_query(query, conn).head(10)


### Wiederholung: GROUP BY

Frage: Wie viele Bestellungen gibt es pro Kundin oder Kunde?


In [ ]:
query = '''
SELECT
    customer_id,
    COUNT(*) AS anzahl_bestellungen
FROM orders
GROUP BY customer_id
ORDER BY anzahl_bestellungen DESC;
'''

pd.read_sql_query(query, conn).head(10)


### Wiederholung: Aggregation mit SUM

Frage: Wie viele Artikel wurden pro Kundin oder Kunde insgesamt bestellt?


In [ ]:
query = '''
SELECT
    customer_id,
    SUM(quantity) AS gesamtmenge
FROM orders
GROUP BY customer_id
ORDER BY gesamtmenge DESC;
'''

pd.read_sql_query(query, conn).head(10)


## 2. WHERE vs. HAVING

`WHERE` filtert einzelne Zeilen **vor** der Gruppierung.  
`HAVING` filtert Gruppen **nach** der Gruppierung.


### WHERE: Zeilen vor der Gruppierung filtern

Frage: Welche Kundinnen und Kunden haben Bestellungen mit mehr als 3 Artikeln aufgegeben?


In [ ]:
query = '''
SELECT
    customer_id,
    COUNT(*) AS anzahl_grosser_bestellungen
FROM orders
WHERE quantity > 3
GROUP BY customer_id
ORDER BY anzahl_grosser_bestellungen DESC;
'''

pd.read_sql_query(query, conn).head(10)


### HAVING: Gruppen nach der Aggregation filtern

Frage: Welche Kundinnen und Kunden haben insgesamt mehr als 8 Artikel bestellt?


In [ ]:
query = '''
SELECT
    customer_id,
    SUM(quantity) AS gesamtmenge
FROM orders
GROUP BY customer_id
HAVING SUM(quantity) > 8
ORDER BY gesamtmenge DESC;
'''

pd.read_sql_query(query, conn)


### Vergleich

- `WHERE quantity > 3`: betrachtet einzelne Bestellzeilen.
- `HAVING SUM(quantity) > 8`: betrachtet das Ergebnis pro Gruppe.

Das ist ein wichtiger Denkwechsel: Erst Zeilen, dann Gruppen.


## 3. JOIN: Orders mit Customers verbinden

Die `orders`-Tabelle enthält nur IDs. Mit einem Join werden daraus verständliche Kundendaten.


In [ ]:
query = '''
SELECT
    o.order_id,
    c.name,
    c.city,
    o.product_id,
    o.quantity
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
ORDER BY o.order_id;
'''

pd.read_sql_query(query, conn).head(10)


### JOIN plus Tag-1-Konzepte

Frage: Wie viele Bestellungen und Artikel gibt es pro Stadt?


In [ ]:
query = '''
SELECT
    c.city,
    COUNT(o.order_id) AS anzahl_bestellungen,
    SUM(o.quantity) AS gesamtmenge
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
GROUP BY c.city
ORDER BY gesamtmenge DESC;
'''

pd.read_sql_query(query, conn)


## 4. Drei Tabellen kombinieren

Für Umsatz brauchen wir zusätzlich den Produktpreis aus `products`.


In [ ]:
query = '''
SELECT
    o.order_id,
    c.name,
    c.city,
    p.product_name,
    p.category,
    o.quantity,
    p.price,
    o.quantity * p.price AS revenue
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
INNER JOIN products AS p
    ON o.product_id = p.product_id
ORDER BY o.order_id;
'''

pd.read_sql_query(query, conn).head(10)


### Umsatz pro Kategorie

Hier verbinden wir Joins mit `GROUP BY`, `SUM` und `ORDER BY`.


In [ ]:
query = '''
SELECT
    p.category,
    SUM(o.quantity * p.price) AS revenue
FROM orders AS o
INNER JOIN products AS p
    ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC;
'''

pd.read_sql_query(query, conn)


## 5. Unterabfrage

Eine Unterabfrage ist eine Abfrage innerhalb einer anderen Abfrage.

Frage: Welche Kundinnen und Kunden haben mehr Artikel bestellt als der Durchschnitt pro Kundin oder Kunde?


In [ ]:
query = '''
SELECT
    customer_id,
    SUM(quantity) AS gesamtmenge
FROM orders
GROUP BY customer_id
HAVING SUM(quantity) > (
    SELECT AVG(kundenmenge)
    FROM (
        SELECT SUM(quantity) AS kundenmenge
        FROM orders
        GROUP BY customer_id
    )
)
ORDER BY gesamtmenge DESC;
'''

pd.read_sql_query(query, conn)


## 6. CTE mit WITH

Eine CTE macht Zwischenergebnisse lesbarer. Die Logik ist ähnlich wie bei der Unterabfrage, aber oft leichter zu erklären und zu debuggen.


In [ ]:
query = '''
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(quantity) AS gesamtmenge
    FROM orders
    GROUP BY customer_id
),
average_total AS (
    SELECT AVG(gesamtmenge) AS durchschnitt
    FROM customer_totals
)
SELECT
    ct.customer_id,
    ct.gesamtmenge
FROM customer_totals AS ct
CROSS JOIN average_total AS avg_t
WHERE ct.gesamtmenge > avg_t.durchschnitt
ORDER BY ct.gesamtmenge DESC;
'''

pd.read_sql_query(query, conn)


### Subquery vs. CTE

Beide Varianten beantworten dieselbe Frage.

- Unterabfragen sind kompakt.
- CTEs sind oft lesbarer, wenn mehrere Zwischenschritte nötig sind.
- Für Unterricht und Teamarbeit sind CTEs häufig leichter zu diskutieren.


## 7. Mini-Visualisierung

Zum Abschluss visualisieren wir den Umsatz pro Produktkategorie.


In [ ]:
query = '''
SELECT
    p.category,
    SUM(o.quantity * p.price) AS revenue
FROM orders AS o
INNER JOIN products AS p
    ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC;
'''

category_revenue = pd.read_sql_query(query, conn)

sns.barplot(data=category_revenue, x='revenue', y='category')
plt.title('Umsatz pro Produktkategorie')
plt.xlabel('Umsatz')
plt.ylabel('Kategorie')
plt.show()


## Zusammenfassung

Heute hast du Tag-1-Grundlagen erweitert:

- `GROUP BY` beantwortet Fragen pro Gruppe.
- `HAVING` filtert aggregierte Gruppen.
- `JOIN` verbindet IDs mit Bedeutung.
- Subqueries und CTEs helfen bei mehrstufigen Analysefragen.
